## Multi-Category HTML Parsing

### Overview
Extends the AC-only parser (notebook 01) to handle **3 appliance categories**: AC, Refrigerator, Washing Machine.
Goal: produce a unified `appliances_parsed.csv` with a `category` column for downstream multi-category modeling.

### Workflow
1. Inspect the structure of `refrigerator.html` and `washing_machines.html` to verify selectors work
2. Print a sample card from each category to discover per-category feature structure
3. Write per-category parser logic (cells 6-7)
4. Concatenate all 3 categories and save the unified CSV

### Why 3 categories, not 4
Microwave is **out of scope** for this pass. Re-evaluate after notebook 10 (multi-category model experiments).

In [ ]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
import sys
sys.path.append("D:/Study/data_science/underpriced-listing-predictor/scraper")

import ac_selectors as s

In [ ]:
html_files = {
    'AC':            'D:/Study/data_science/underpriced-listing-predictor/data/01.raw/ac_full_dump.html',
    'Refrigerator':  'D:/Study/data_science/underpriced-listing-predictor/data/01.raw/refrigerator.html',
    'WashingMachine':'D:/Study/data_science/underpriced-listing-predictor/data/01.raw/washing_machines.html',
}
# NOTE: Microwave is out of scope for this pass. Add later if needed.

for label, path in html_files.items():
    with open(path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'lxml')
    cards = soup.find_all('div', {'class':'sm-product has-tag has-features has-actions'})
    print(f"{label}: {len(cards)} cards")

In [ ]:
for label, path in html_files.items():
    with open(path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'lxml')
    cards = soup.find_all('div', {'class':'sm-product has-tag has-features has-actions'})
    if not cards:
        print(f"\n=== {label}: NO CARDS FOUND ===")
        # Fallback: see what class strings actually exist
        all_sm = soup.find_all('div', {'class': re.compile(r'sm-product')})
        if all_sm:
            print(f"  Found {len(all_sm)} divs matching 'sm-product'. Sample class string:")
            print(f"  '{all_sm[0].get('class')}'")
        continue
    sample = cards[0]
    print(f"\n=== {label} sample card ===")
    print(f"Name: {sample.find('h2').text}")
    print(f"Price: {sample.find('span', {'class':'price'}).text}")
    features = sample.find('ul', {'class':'sm-feat specs'}).find_all('li')
    print(f"Features ({len(features)}):")
    for i, f in enumerate(features):
        print(f"  [{i}] {f.text}")

### Per-category parser design (filled in from cell 4 output)

**Discovered structure** (capacity position varies!):

| Category      | [0]              | [1]                | [2]      | [3]        | [4]-[7] sub-features       |
|---------------|------------------|--------------------|----------|------------|----------------------------|
| AC            | ac_type          | capacity (ton)     | —        | —          | 6 sub-features             |
| Refrigerator  | door_type        | freezer_position   | capacity (L) | energy_rating | 4 sub-features         |
| WashingMachine| dryer_flag       | load_type          | capacity (kg) | —      | 5 sub-features             |

**Output schema** (uniform across all 3 categories):

| Column            | Type   | Example (AC)            | Example (Fridge)        | Example (WM)        |
|-------------------|--------|-------------------------|-------------------------|---------------------|
| `name`            | str    | "Whirlpool SAI18B..."   | "Samsung RT28C..."      | "Samsung Ecobub..." |
| `price`           | str    | "₹24,990"               | "₹25,490"               | "₹13,294"           |
| `rating`          | str    | "--rating: 4.65;"       | "--rating: 4.3;"        | "--rating: 4.4;"    |
| `category`        | str    | "AC"                    | "Refrigerator"          | "WashingMachine"    |
| `type_field`      | str    | "Split AC"              | "Multi Door"            | "Top Load"          |
| `capacity_raw`    | str    | "1.5 Ton"               | "236 L"                 | "8 kg"              |
| `energy_rating`   | str    | "5 Star" (from subfeat) | "3 Star" (from [3])     | NaN                 |
| `sub_features`    | list   | [Air Swing, ...]        | [Toughened Glass, ...]  | [PP Dual Wing, ...] |

**Why this works**:
- `type_field` is the most specific *category-conditional* descriptor (used for one-hot encoding later)
- `capacity_raw` keeps the unit (Ton / L / kg) — unit normalization happens in notebook 08
- `energy_rating` is in the dedicated position for fridge, scraped from sub-features for AC, NaN for WM (no star rating)
- `sub_features` catches the rest — these get one-hot extracted in notebook 08 after you see the full vocabulary

**Note on parsing**: each category has its own field positions, so `parse_category()` is category-aware. Use an `if category == 'AC':` style branch (or 3 separate small functions).

In [ ]:
# Per-category parsers — write this after running cell 4 and filling in cell 5
# Recommended signature: parse_category(html_path, label) -> pd.DataFrame
# Output columns: name, price, rating, category, cat_feature_1, capacity_raw, sub_features (list)

def parse_category(html_path, label):
    raise NotImplementedError("Write per-category parser logic here based on cell 4 output")

In [ ]:
# Run all 3 parsers, concatenate, save
# dfs = []
# for label, path in html_files.items():
#     df = parse_category(path, label)
#     dfs.append(df)
#
# all_appliances = pd.concat(dfs, ignore_index=True)
# print(f"Total: {len(all_appliances)} rows")
# print(all_appliances['category'].value_counts())
#
# all_appliances.to_csv(
#     'D:/Study/data_science/underpriced-listing-predictor/data/02.parsed/appliances_parsed.csv',
#     index=False
# )

### Next steps

Once `appliances_parsed.csv` is saved, the pipeline continues:

- **08.ac_multi_category_cleaning.ipynb** — unit normalization (tons→kW? liters→L, kg→kg), missing values, brand extraction, per-category feature engineering
- **09.ac_multi_category_eda.ipynb** — combined EDA with `category` as a feature, check for category-conditional price distributions
- **10.ac_multi_category_model_experiments.ipynb** — re-run 9-model comparison on combined data (target: R² ≥ 0.70)
- **11.ac_multi_category_shap.ipynb** — SHAP on the multi-category winner
- **12.ac_streamlit_app.ipynb** — UI with category selector